In [0]:
dbutils.widgets.text("p_batch_id", ' ')
v_batch_id =  dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.Environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/sprints"
table_name = f"{catalog_name}.{bronze_schema}.sprints"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

sprints_schema = StructType([
    StructField("date", DateType(), True),
    StructField("raceName", StringType(), True),
    StructField("round", IntegerType(), True),
    StructField("season", IntegerType(), True),
    StructField("url", StringType(), True),
    StructField("constructorId", StringType(), True),
    StructField("driverId", StringType(), True),
    StructField("grid", IntegerType(), True),
    StructField("laps", IntegerType(), True),
    StructField("number", IntegerType(), True),
    StructField("points", DoubleType(), True),
    StructField("position", IntegerType(), True),
    StructField("positionText", StringType(), True),
    StructField("status", StringType(), True)
])

In [0]:
sprints_df = (
    spark.read
        .format('json')
        .option('mode','FAILFAST')
        .option('multiLine',True)
        .schema(sprints_schema)
        .load(source_file)
)

In [0]:
display(sprints_df)

In [0]:
sprints_final_df = add_ingestion_metadata(sprints_df)

In [0]:
write_to_bronze(
    input_df = sprints_final_df,
    target_table = table_name,
    batch_id = v_batch_id
)

In [0]:
display(spark.table(table_name))